<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">دو گذشته، یک <bdi dir="ltr">Token</bdi> آخر</h1>
<p style="text-align:right">درس 31 از 92 · از <bdi dir="ltr">MLP</bdi> تا مدل دنباله: گذشته از کجا وارد می‌شود؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">26a-sequence-models</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-01/26a-sequence-models.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">نشان دهید ترتیب پنجره کجا حفظ می‌شود و کجا از دست می‌رود.</p><p style="text-align:right"><span class="phrase-lead" style="white-space:nowrap">پیش‌نیاز: ورودی</span> سه‌محوری، <bdi dir="ltr">Linear</bdi> و نسخهٔ <bdi dir="ltr">v1</bdi> را بشناسید.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۳۰–۵۵ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">دو پنجره از بردارهای یکسان ولی با ترتیب معکوس داریم. میانگین آن‌ها و خروجی یک خوانش خطیِ پنجره‌ای را پیش‌بینی کنید.</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.stages.v1 import TokenOnly
windows = torch.tensor([[[1.,0.],[0.,1.]], [[0.,1.],[1.,0.]]])
weight = torch.tensor([[1.,0.,0.,2.]])
model = TokenOnly(5, channels=3).eval()
ids = torch.tensor([[1,3],[2,3]])
with torch.no_grad():
    print("v1 last-position difference:", (model(ids)[0,-1]-model(ids)[1,-1]).abs().max().item())
print("windows:", windows, "pooled:", windows.mean(1))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">window_readout(x, weight)</code> را بنویسید: <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">x</code> شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,T,C)</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">weight</code> شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(O,T*C)</code> دارد؛ خروجی باید <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,O)</code> و بدون <bdi dir="ltr">Bias</bdi> باشد. فقط محورهای زمان و ویژگی را صاف کنید.</p>
</div>

In [ ]:
def window_readout(x, weight):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = window_readout(windows, weight)
    if result is None: return False
    torch.testing.assert_close(result, torch.tensor([[3.],[0.]]))
    other = torch.arange(24.).reshape(2,3,4)
    w = torch.arange(24.).reshape(2,12)/10
    expected = torch.stack([torch.stack([(sample.reshape(-1)*row).sum() for row in w]) for sample in other])
    torch.testing.assert_close(window_readout(other,w), expected)
    assert window_readout(other,w).shape == (2,2)
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط جای دو موقعیت را عوض کنید؛ وزن و مجموعهٔ بردارها ثابت بمانند. چرا خوانش پنجره‌ای تغییر می‌کند ولی میانگین نه؟</p>
</div>

In [ ]:
print('original:', windows[0].flatten() @ weight[0])
print('reversed:', windows[0].flip(0).flatten() @ weight[0])
print('same mean:', torch.equal(windows[0].mean(0), windows[0].flip(0).mean(0)))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">نسخهٔ خراب ترتیب را پیش از خوانش با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">mean</code> حذف کرده است. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">ordered_features(x)</code> را اصلاح کنید تا <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,T*C)</code> بدهد؛ با عوض‌کردن وزن، اطلاعات حذف‌شده برنمی‌گردد.</p>
</div>

In [ ]:
wrong_features = windows.mean(1)
print('different windows, identical features:', torch.equal(wrong_features[0], wrong_features[1]))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def ordered_features(x):
    # TODO
    return None

In [ ]:
def test_repair():
    result = ordered_features(windows)
    if result is None: return False
    assert result.shape == (2,4)
    assert not torch.equal(result[0],result[1])
    assert torch.equal(ordered_features(torch.arange(24).reshape(2,3,4)),torch.arange(24).reshape(2,12))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">TokenOnly</code> واقعی در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">mini_gpt/stages/v1.py</code> هیچ مسیر بین موقعیت‌ها ندارد. خوانش پنجره‌ای این تمرین یک آزمایش کوچک است، نه جایگزین معماری <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT</code>؛ <bdi dir="ltr">Attention</bdi> در مرحلهٔ بعد این محدودیت را به روش دیگری برطرف می‌کند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">چه اطلاعاتی از گذشته باید در ساختن خروجیِ آخر نقش داشته باشد؟ چرا نمی‌توان محدودیت <bdi dir="ltr">v1</bdi> را به همهٔ <bdi dir="ltr">MLP</bdi>ها نسبت داد؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-01/26a-sequence-models.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/26a-sequence-models.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>